# APC-Vis 7B · perception ablation · 02_orientanything_v2

Kaggle: **GPU T4 ×2**, Internet ON, Secret **HF_token** with SAM 3 access.
Add the `sc203-apc-trace120` dataset prepared alongside these notebooks. Set `trace_input`
only if auto-detection finds several bundles. Code is cloned from public sc203-kat.

This is **controlled replay**, not a new full-dataset baseline: preserve the historical
object descriptions, selected viewer and Depth Pro map. Recompute downstream geometry
when a selected box/mask changes; rerender and query the same Qwen 7B in FP16.

Baseline results are imported from the existing trace bundle; baseline inference is never run.
Sanity runs all three branches. Full notebook 01 runs SAM 3 + OneRef; notebook 02 runs OA V2. Optional combinations: `sam3_oa2`, `oneref_oa2`, `sam3_oneref`,
`sam3_oneref_oa2`. Use a new output directory for
a changed experiment. Sanity uses one image; each full notebook uses all supplied diagnostic traces for its branches.

OneRef sees the original object description and full image, not the requested spatial
answer. It ranks the same top-5 proposals by overlap with its predicted REC box; it cannot
recover an instance missing from those proposals. No hidden fallback to object 0.

Runtime environments/weights stay in `/kaggle/temp`; evidence stays in `/kaggle/working`.
Optional `weight_inputs` avoid auxiliary downloads. `model_input` is the existing APC
asset bundle, not just Qwen weights. To resume, add the previous Output as Input and set
`resume_input` to its `apc_ablation_02_orientanything_v2_v3` folder. Only identical code/config/cohort resume.


In [ ]:
REPO_URL = 'https://github.com/leetranwyng/sc203-kat.git'
CODE_REF = 'v0.3.0-ablation.3'
HF_SECRET_NAME = 'HF_token'
RUN_CONFIG = {'method': 'apc_vis',
 'mode': 'full',
 'study': '02_orientanything_v2',
 'trace_input': None,
 'model_input': None,
 'resume_input': None,
 'output_dir': '/kaggle/working/apc_ablation_02_orientanything_v2_v3',
 'runtime_dir': '/kaggle/temp/apc_ablation_runtime/02_orientanything_v2',
 'variants': ['oa2'],
 'sample_ids': None,
 'weight_inputs': {'sam3': None, 'oneref': None, 'oneref_tokenizer': None, 'oa2': None},
 'sam3_threshold': 0.5,
 'oneref_min_iou': 0.0,
 'seed': 42,
 'sample_timeout_seconds': 900,
 'phase_timeout_seconds': 21600,
 'disk_min_free_gib': 3.0,
 'retry_errors': False}


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

if not CODE_REF or CODE_REF.startswith('-') or any(c.isspace() for c in CODE_REF):
    raise ValueError('CODE_REF must be a Git tag, branch, or full commit.')
if REPO_URL != 'https://github.com/leetranwyng/sc203-kat.git':
    raise ValueError('Use the public leetranwyng/sc203-kat repository.')
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT='0', GCM_INTERACTIVE='Never')
cache = Path(os.environ.get('SC203_CODE_CACHE', '/kaggle/temp/sc203-kat-code'))
cache.mkdir(parents=True, exist_ok=True)
key = hashlib.sha256((REPO_URL + '\n' + CODE_REF).encode()).hexdigest()[:16]
CODE_ROOT = cache / key

def git(*args):
    result = subprocess.run(['git', *args], env=GIT_ENV, text=True,
                            capture_output=True, timeout=300)
    if result.returncode: raise RuntimeError((result.stderr or result.stdout).strip())
    return result.stdout.strip()

if not (CODE_ROOT / '.git').exists():
    if CODE_ROOT.exists() and any(CODE_ROOT.iterdir()):
        raise RuntimeError('Code cache is not empty; choose a different SC203_CODE_CACHE.')
    git('clone', '--no-checkout', '--no-tags', '--depth', '1', REPO_URL, str(CODE_ROOT))
else:
    if git('-C', str(CODE_ROOT), 'remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('Cached source origin differs from REPO_URL.')
    if git('-C', str(CODE_ROOT), 'status', '--porcelain', '--untracked-files=normal'):
        raise RuntimeError('Cached source has local changes; use a new code cache.')
git('-C', str(CODE_ROOT), 'fetch', '--depth', '1', '--no-tags', 'origin', CODE_REF)
commit = git('-C', str(CODE_ROOT), 'rev-parse', '--verify', 'FETCH_HEAD^{commit}')
git('-C', str(CODE_ROOT), 'checkout', '--detach', commit)

ENV = dict(GIT_ENV, SC203_KAT_ROOT=str(CODE_ROOT), PYTHONPATH=str(CODE_ROOT / 'src'))
if HF_SECRET_NAME and not ENV.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(HF_SECRET_NAME)
        if value: ENV['HF_TOKEN'] = value.strip()
        del value
    except Exception:
        print('HF secret unavailable; using public/local model files.')
config_dir = Path(os.environ.get('SC203_CONFIG_DIR', '/kaggle/temp/sc203-kat-configs'))
config_dir.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = config_dir / (RUN_CONFIG['method'] + '_' + RUN_CONFIG['mode'] + '.json')
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
prepare = (
    'import json,sys; from pathlib import Path; '
    'from sc203_kat.runtime.kaggle import ensure_submodule,source_metadata; '
    'root=Path(sys.argv[1]); method=sys.argv[2]; '
    'ensure_submodule(root,method); metadata=source_metadata(root,method); '
    'print(json.dumps(metadata,indent=2))'
)
subprocess.run([sys.executable, '-c', prepare, str(CODE_ROOT), RUN_CONFIG['method']], cwd=CODE_ROOT, env=ENV, check=True)
print('Code:', commit, '| Method:', RUN_CONFIG['method'], '| Mode:', RUN_CONFIG['mode'])


In [ ]:
# This orchestration process does not load ML models in the notebook kernel.
subprocess.run([sys.executable, '-u', '-m', 'sc203_kat.ablation.run', '--config', str(CONFIG_PATH)],
               env=ENV, cwd=CODE_ROOT, check=True)


In [ ]:
from IPython.display import FileLink, display
output = Path(RUN_CONFIG['output_dir'])
if (output / 'summary.json').exists():
    summary = json.loads((output / 'summary.json').read_text())
    import pandas as pd
    display(pd.DataFrame(summary['by_task']))
    display(pd.DataFrame(summary['pairs']))
    display(FileLink(str(output / 'index.html')))
    print('Download the complete output folder to keep the HTML images and raw traces.')
    print('Notebook finished != sanity PASS: inspect error status and each adapter output before the full notebooks.')
else:
    print('No summary yet. Inspect the setup cell and logs; rerun after fixing the reported failure.')


In [ ]:
# Optional: archive RESULTS only; source code always comes from GitHub.
# Keep false to avoid storing a second copy when Kaggle disk is low.
EXPORT_RESULT_ZIP = False
if EXPORT_RESULT_ZIP:
    import shutil
    if shutil.disk_usage(output).free < 2 * sum(p.stat().st_size for p in output.rglob('*') if p.is_file()):
        raise RuntimeError('Not enough disk to duplicate evidence as ZIP; download the output folder instead.')
    archive = shutil.make_archive(str(output), 'zip', root_dir=output.parent, base_dir=output.name)
    display(FileLink(archive))
